# CivicPulse — Notebook 03: Classification, Clustering, Evaluation & Deployment
**Course Outcome Coverage:** CO3 (Multi-task Classification: Spam, Category, Severity; Topic Modeling with LDA & KMeans; Coherence & Silhouette Scores; Comprehensive Evaluation: Precision/Recall/F1, Confusion Matrices, ROC/AUC, Error Analysis; FastAPI Deployment & Docker Architecture)
**Author:** CivicPulse Capstone Team
**Date:** October 2026

---

## 1. Multi-Task Classification Architecture
CivicPulse executes three sequential classification decisions upon ingestion:
1. **Spam & Clickbait Filtering:** Discards real estate ads and viral news before alert creation.
2. **Alert Category Classification:** Directs the alert to the appropriate municipal department (road, water, power, weather, public service, other).
3. **Urgency / Severity Classification:** Prioritizes push notifications and visual prominence (`info` -> `advisory` -> `warning` -> `critical`).

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score

from server.nlp.classifiers import model_suite
from server.nlp.clustering import CivicTopicModeler, find_duplicate_reports
from server.nlp.preprocess import default_preprocessor

# Load production model suite
model_suite.load_artifacts()
print("Model suite loaded.")

## 2. Topic Modeling (LDA) & Deduplication Clustering

### Unsupervised Topic Discovery with Latent Dirichlet Allocation (LDA)
LDA models each document as a Dirichlet mixture of topics, and each topic as a distribution over vocabulary words. This allows unsupervised discovery of emerging civic crises without manual tagging.

In [ ]:
with open(PROJECT_ROOT / "seed" / "seed_data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

civic_reports = [d["text"] for d in dataset if not d.get("is_spam", False)]

topic_modeler = CivicTopicModeler(n_topics=5)
lda_results = topic_modeler.fit_lda(civic_reports)

print(f"LDA Perplexity: {lda_results['perplexity']}")
print(f"Topic Keyword Diversity: {lda_results['topic_diversity']}")
print("\nDiscovered Civic Topics:")
for t in lda_results["topics"]:
    print(f"  Topic #{t['topic_id']}: {', '.join(t['keywords'][:6])}")

### Deduplication Logic Demonstration
When multiple citizens or media outlets report the same pipe burst or fallen tree, CivicPulse groups near-duplicates using pairwise cosine distance over TF-IDF unigrams:

In [ ]:
duplicate_samples = [
    "Water supply will remain interrupted in Begumpet due to pipeline maintenance work.",
    "Pipeline burst in Begumpet has interrupted municipal drinking water supply.",
    "Major traffic blockade on Outer Ring Road due to multi-car accident.",
    "Outer Ring Road blocked near flyover following a road accident."
]

clusters = find_duplicate_reports(duplicate_samples, similarity_threshold=0.50)
print("Deduplication Clusters:")
for idx, group in enumerate(clusters, 1):
    print(f"  Incident #{idx}:")
    for i in group:
        print(f"    - {duplicate_samples[i]}")

## 3. Comprehensive Model Evaluation on Test Set

In [ ]:
with open(PROJECT_ROOT / "seed" / "test.json", "r", encoding="utf-8") as f:
    test_set = json.load(f)

test_non_spam = [d for d in test_set if not d.get("is_spam", False)]
y_true_cat = [d["category"] for d in test_non_spam]
y_true_sev = [d["severity"] for d in test_non_spam]

preds = [model_suite.predict(d["text"]) for d in test_non_spam]
y_pred_cat = [p["category"] for p in preds]
y_pred_sev = [p["severity"] for p in preds]

print("=== Category Classification Report (Test Set) ===")
print(classification_report(y_true_cat, y_pred_cat))

print("=== Severity Classification Report (Test Set) ===")
print(classification_report(y_true_sev, y_pred_sev))

## 4. Confusion Matrices & ROC Curves

In [ ]:
from IPython.display import Image, display

cm_path = PROJECT_ROOT / "reports" / "confusion_matrices" / "category_confusion_matrix.png"
roc_path = PROJECT_ROOT / "reports" / "roc_curves" / "multiclass_ovr_roc_curves.png"

print("Category Confusion Matrix:")
if cm_path.exists():
    display(Image(filename=str(cm_path)))

print("Multiclass One-vs-Rest ROC Curves:")
if roc_path.exists():
    display(Image(filename=str(roc_path)))

## 5. Deployment Architecture: FastAPI & Docker Basics

### FastAPI Microservice Design
- **Endpoints:** `/api/alerts`, `/api/alerts/{id}/explain`, `/api/ingest`, `/api/classify`, `/api/topics`, `/health`.
- **Validation:** Pydantic models with automated schema enforcement and HTTP 422 error handlers.
- **Rate Limiting:** IP-keyed rate limits via SlowAPI (`60 requests/minute`).
- **Persistence:** SQLite ORM via SQLAlchemy with thread-safe connection pooling.

### Docker Basics
- **Image:** A self-contained, immutable snapshot containing Python 3.11, PyTorch, spaCy models, and application code.
- **Container:** An isolated running instance of the image communicating over internal Docker bridge networks.
- **Docker Compose:** Orchestrates two services: `api` (FastAPI on port 8000) and `client` (Vite SPA on port 3000).